In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import re
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV


# ============================================================
# FONCTIONS FEATURES
# ============================================================

UNIT_PATTERN = r"(cm|mm|m|kg|g|mg|l|ml|cl|w|kw|v|mah|ah|hz|ghz|mhz|go|gb|to|tb|mp|px|fps|°c|°)"

def get_designation(X):
    return X["designation"].fillna("").astype(str)

def get_description(X):
    return X["description"].fillna("").astype(str)

def first_words_series(X, n=3):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.split()
        .str[:n]
        .str.join(" ")
    )

def numbers_units_series(X):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.findall(rf"\b\d+[.,]?\d*\s?{UNIT_PATTERN}\b")
        .str.join(" ")
    )


# ============================================================
# CHARGEMENT DONNÉES
# ============================================================

X_path = r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\FEV26-CMLOPS-RAKUTEN\data\raw\X_train_update.csv"
Y_path = r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\FEV26-CMLOPS-RAKUTEN\data\processed\Y_train_encode.csv"

X = pd.read_csv(X_path)
Y = pd.read_csv(Y_path)

df = X.merge(Y, on="Unnamed: 0")

print("Dataset shape :", df.shape)


# ============================================================
# X / y
# ============================================================

y = df["prdtypecode_encoded"]
X = df.drop(columns=["prdtypecode_encoded"])


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# TF-IDF
# ============================================================

word_tfidf = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    lowercase=True
)


# ============================================================
# FEATURES
# ============================================================

features = ColumnTransformer([

    ("designation_word",
     Pipeline([
         ("select", FunctionTransformer(get_designation, validate=False)),
         ("tfidf", word_tfidf)
     ]),
     ["designation"]),

    ("designation_char",
     Pipeline([
         ("select", FunctionTransformer(get_designation, validate=False)),
         ("tfidf", char_tfidf)
     ]),
     ["designation"]),

    ("description_word",
     Pipeline([
         ("select", FunctionTransformer(get_description, validate=False)),
         ("tfidf", word_tfidf)
     ]),
     ["description"]),

    ("description_char",
     Pipeline([
         ("select", FunctionTransformer(get_description, validate=False)),
         ("tfidf", char_tfidf)
     ]),
     ["description"]),

    ("first_words",
     Pipeline([
         ("extract", FunctionTransformer(first_words_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),

    ("numbers_units",
     Pipeline([
         ("extract", FunctionTransformer(numbers_units_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),
])


# ============================================================
# PIPELINE
# ============================================================

pipe = Pipeline([
    ("features", features),
    ("clf", LinearSVC(max_iter=20000, class_weight="balanced"))
])


# ============================================================
# PARAM SEARCH
# ============================================================

param_dist = {

    "features__designation_word__tfidf__max_features": [20000,30000,40000,50000],
    "features__description_word__tfidf__max_features": [20000,30000,40000,50000],

    "features__designation_word__tfidf__ngram_range": [(1,1),(1,2)],
    "features__description_word__tfidf__ngram_range": [(1,1),(1,2)],

    "features__designation_char__tfidf__ngram_range": [(3,5),(3,6)],
    "features__description_char__tfidf__ngram_range": [(3,5),(3,6)],

    "features__designation_char__tfidf__min_df": [2,3],
    "features__description_char__tfidf__min_df": [2,3],

    "clf__C": [0.5,0.8,1,1.2,1.5,2]
}


# ============================================================
# RANDOM SEARCH
# ============================================================

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="f1_weighted",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("==> Lancement recherche paramètres...")

search.fit(X_train, y_train)


# ============================================================
# RESULTATS
# ============================================================

print("\n🏆 Best parameters :")
print(search.best_params_)

print("\n🏆 Best CV score :")
print(search.best_score_)


# ============================================================
# TEST FINAL
# ============================================================

best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

print("\nF1 weighted test :", f1_score(y_test, y_pred, average="weighted"))

Dataset shape : (84916, 8)
==> Lancement recherche paramètres...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

🏆 Best parameters :
{'features__designation_word__tfidf__ngram_range': (1, 2), 'features__designation_word__tfidf__max_features': 50000, 'features__designation_char__tfidf__ngram_range': (3, 5), 'features__designation_char__tfidf__min_df': 2, 'features__description_word__tfidf__ngram_range': (1, 2), 'features__description_word__tfidf__max_features': 30000, 'features__description_char__tfidf__ngram_range': (3, 6), 'features__description_char__tfidf__min_df': 3, 'clf__C': 1}

🏆 Best CV score :
0.8548000694289885

F1 weighted test : 0.8665983252880718
